In [5]:
import numpy as np

def calculate_lswi(b02, b06, qa_good):
    nir = np.asarray(b02, dtype=np.float32) * 0.0001
    swir = np.asarray(b06, dtype=np.float32) * 0.0001

    valid = qa_good & (b02 != -28672) & (b06 != -28672) & ((nir + swir) != 0)

    result = np.full(nir.shape, np.nan, dtype=np.float32)
    result[valid] = (nir[valid] - swir[valid]) / (nir[valid] + swir[valid])
    return result

In [6]:
from pathlib import Path
import re

import ipywidgets as widgets
import numpy as np
import pandas as pd
import rasterio
from IPython.display import clear_output, display
from ipyleaflet import Map, Marker
from pyproj import Transformer

search_locations = [Path.cwd(), *Path.cwd().parents]
lswi_roots = []
for location in search_locations:
    lswi_roots.extend([
        location / 'data' / 'datacollect' / 'nasa' / 'lswi',
        location / 'lswi',
    ])
    if location.name.lower() == 'lswi':
        lswi_roots.append(location)
dataset_dirs = {}
for root in dict.fromkeys(lswi_roots):
    if not root.is_dir():
        continue
    for folder in root.iterdir():
        if folder.is_dir() and next(folder.glob('*_b02_*.tif'), None) is not None:
            label = {'data': 'Bangladesh', 'egypt_data': 'Egypt'}.get(
                folder.name.lower(), folder.name.replace('_data', '').replace('_', ' ').title()
            )
            dataset_dirs[label] = folder
if not dataset_dirs:
    raise FileNotFoundError('Could not find any LSWI region folders containing *_b02_*.tif')

def load_dataset(data_dir):
    assets_by_date = {}
    for path in data_dir.glob('*.tif'):
        match = re.search(r'_(b02|b06|qc_500m)_([0-9]{8})T', path.name)
        if match:
            band, date = match.groups()
            assets_by_date.setdefault(date, {})[band] = path

    paired = {
        date: assets for date, assets in sorted(assets_by_date.items())
        if all(band in assets for band in ('b02', 'b06', 'qc_500m'))
    }
    if not paired:
        raise FileNotFoundError(f'No dates with matching B02, B06, and QC rasters in {data_dir}')

    first = next(iter(paired.values()))['b02']
    with rasterio.open(first) as reference:
        crs = reference.crs
        transform = reference.transform
        shape = (reference.height, reference.width)
        center_x, center_y = reference.xy(reference.height // 2, reference.width // 2)
    center_lon, center_lat = Transformer.from_crs(
        crs, 'EPSG:4326', always_xy=True
    ).transform(center_x, center_y)

    for assets in paired.values():
        for path in assets.values():
            with rasterio.open(path) as src:
                if src.crs != crs or src.transform != transform or (src.height, src.width) != shape:
                    raise ValueError(f'Raster grid does not match the dataset reference grid: {path}')
    return {
        'directory': data_dir,
        'assets': paired,
        'first_asset': first,
        'crs': crs,
        'coordinate_transformer': Transformer.from_crs('EPSG:4326', crs, always_xy=True),
        'pixel_to_lonlat': Transformer.from_crs(crs, 'EPSG:4326', always_xy=True),
        'center': (center_lat, center_lon),
    }

default_region = 'Bangladesh' if 'Bangladesh' in dataset_dirs else next(iter(dataset_dirs))
dataset_state = load_dataset(dataset_dirs[default_region])
region_selector = widgets.Dropdown(
    options=list(dataset_dirs), value=default_region, description='Region:'
)
lswi_map = Map(center=dataset_state['center'], zoom=6)
selection_output = widgets.Output()
selection_marker = {'value': None}

def classify_lswi(value):
    if not np.isfinite(value):
        return 'No data / invalid pixel'
    if value <= -0.1:
        return '≤ -0.1: Extreme drought / desiccation — severe water stress'
    if value < 0.2:
        return '-0.1 to 0.2: Low moisture / soil saturated or bare background'
    if value < 0.6:
        return '0.2 to 0.6: Healthy green vegetation'
    if value <= 1.0:
        return '0.6 to 1.0: Waterlogged / saturated surface water'
    return 'Outside the listed LSWI ranges'

def show_lswi_at_click(**event):
    if event.get('type') != 'click':
        return

    latitude, longitude = event['coordinates']
    x, y = dataset_state['coordinate_transformer'].transform(longitude, latitude)
    with rasterio.open(dataset_state['first_asset']) as src:
        row, col = src.index(x, y)
        if not (0 <= row < src.height and 0 <= col < src.width):
            with selection_output:
                clear_output(wait=True)
                print(f'({latitude:.5f}, {longitude:.5f}) is outside the raster coverage.')
            return

    with rasterio.open(dataset_state['first_asset']) as src:
        pixel_x, pixel_y = src.xy(row, col)
    pixel_lon, pixel_lat = dataset_state['pixel_to_lonlat'].transform(pixel_x, pixel_y)

    lswi_values = []
    window = ((row, row + 1), (col, col + 1))
    for date, assets in dataset_state['assets'].items():
        with rasterio.open(assets['b02']) as nir_src, rasterio.open(assets['b06']) as swir_src, rasterio.open(assets['qc_500m']) as qc_src:
            nir = int(nir_src.read(1, window=window)[0, 0])
            swir = int(swir_src.read(1, window=window)[0, 0])
            qc = int(qc_src.read(1, window=window)[0, 0])

        quality_ok = (qc & 0b11) == 0 and ((qc >> 6) & 0b1111) == 0 and ((qc >> 22) & 0b1111) == 0
        value = calculate_lswi(np.array([nir]), np.array([swir]), np.array([quality_ok]))[0]
        lswi_value = float(value) if np.isfinite(value) else np.nan
        if np.isfinite(lswi_value):
            lswi_values.append(lswi_value)

    mean_lswi = float(np.mean(lswi_values)) if lswi_values else np.nan
    summary = pd.DataFrame([{
        'Mean LSWI': mean_lswi,
        'LSWI class': classify_lswi(mean_lswi),
        'Valid dates': len(lswi_values),
    }])

    if selection_marker['value'] is not None:
        lswi_map.remove_layer(selection_marker['value'])
    selection_marker['value'] = Marker(location=(latitude, longitude))
    lswi_map.add_layer(selection_marker['value'])

    with selection_output:
        clear_output(wait=True)
        print(f'Region: {region_selector.value}')
        print(f'Selected coordinate: latitude {latitude:.5f}, longitude {longitude:.5f}')
        print(f'Raster pixel: row {row}, column {col}; pixel center: latitude {pixel_lat:.5f}, longitude {pixel_lon:.5f}')
        display(summary)

def change_region(change):
    if change['name'] != 'value' or change['new'] == change['old']:
        return
    dataset_state.update(load_dataset(dataset_dirs[change['new']]))
    if selection_marker['value'] is not None:
        lswi_map.remove_layer(selection_marker['value'])
        selection_marker['value'] = None
    lswi_map.center = dataset_state['center']
    lswi_map.zoom = 6
    with selection_output:
        clear_output(wait=True)
        print(f'{change["new"]} selected. Click the map to calculate its mean LSWI.')

region_selector.observe(change_region, names='value')
lswi_map.on_interaction(show_lswi_at_click)
display(region_selector)
display(widgets.HTML('<b>Click the map to get LSWI values for that coordinate across all dates.</b>'))
display(lswi_map, selection_output)


Dropdown(description='Region:', options=('Bangladesh', 'Egypt'), value='Bangladesh')

HTML(value='<b>Click the map to get LSWI values for that coordinate across all dates.</b>')

Map(center=[23.63124999773616, 90.32193113046952], controls=(ZoomControl(options=['position', 'zoom_in_text', …

Output()

## LSWI interpretation ranges

- **≤ -0.1 (Extreme Drought / Desiccation):** Indicates severe water stress.
- **> -0.1 to < 0.2 (Low Moisture / Soil Saturated / Bare Background):** Very dry vegetation, bare soil backgrounds, or early moisture distress.
- **0.2 to < 0.6 (Healthy Green Vegetation):** Standard healthy range for crop canopies and green forests.
- **0.6 to 1.0 (Waterlogged / Saturated Surface Water):** Flooded land, rice paddies, wetlands, or open water bodies.

These are indicative thresholds; validate them for the specific product, region, season, and land cover before treating them as definitive classifications.
